In [1]:

import requests
from bs4 import BeautifulSoup
from pprint import pprint
from selenium import webdriver
from selenium.webdriver.common.by import By
# the module is provided by the webdriver-manager package installed above
from webdriver_manager.chrome import ChromeDriverManager
from lxml import html
from tqdm import tqdm
import time
import pandas as pd
import json

### Scrapping URL's

In [ ]:
# Base configuration
task = 'text-generation'

model_url = []
model_creator = []
model_name = []

# Updated URL (removed the search parameter)
base_url = 'https://huggingface.co/models?pipeline_tag={}&p={}&sort=trending'

# Loop through pages
for page_num in tqdm(range(0, 35)):
    url = base_url.format(task, page_num)
    response = requests.get(url)
    parsed_html = BeautifulSoup(response.text, 'html.parser')
    
    # Extract models
    for tag in parsed_html.body.find_all('article', attrs={'class': 'overview-card-wrapper group/repo'}):
        tag_found = tag.find('a')['href'].strip()
        model_url.append('huggingface.co' + tag_found)
        model_creator.append(tag_found.split('/')[1])
        model_name.append(tag_found.split('/')[2])
        time.sleep(0.5)

# Optional: Save to CSV
df = pd.DataFrame({
    'creator': model_creator,
    'model_name': model_name,
    'url': model_url
})

df.to_csv('huggingface_models.csv', index=False)
print("Scraping completed and saved to huggingface_models.csv")

  0%|          | 0/35 [00:00<?, ?it/s]

  3%|▎         | 1/35 [00:21<11:59, 21.16s/it]


KeyboardInterrupt: 

: 

### Model Cards

In [5]:
# Load the CSV generated from the previous scraping step
df = pd.read_csv("huggingface_models.csv")

model_cards = []

# Iterate through each model URL
for url in tqdm(df["url"], desc="Scraping model cards"):
    full_url = "https://" + url if not url.startswith("https://") else url
    try:
        response = requests.get(full_url, timeout=10)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html.parser")

            # Find the model card content
            card_div = soup.find("div", class_="model-card-content")
            if card_div:
                # Clean and normalize text
                card_text = card_div.get_text(separator=" ").strip()
                card_text = " ".join(card_text.split())  # collapse extra spaces
            else:
                card_text = ""
        else:
            card_text = ""
    except Exception as e:
        print(f"Error for {full_url}: {e}")
        card_text = ""
    
    model_cards.append(card_text)
    time.sleep(0.5)  # respectful delay to avoid rate limiting

# Add the scraped model card text to the dataframe
df["model_card"] = model_cards

# Save updated CSV
df.to_csv("huggingface_models_with_cards.csv", index=False, encoding="utf-8-sig")

print("Scraping complete! Saved as 'huggingface_models_with_cards.csv'")


Scraping model cards:  12%|█▏        | 125/1050 [01:38<11:26,  1.35it/s]

Error for https://huggingface.co/TeichAI/GLM-4.7-Flash-Claude-Opus-4.5-High-Reasoning-Distill: HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)


Scraping model cards:  32%|███▏      | 341/1050 [04:03<07:33,  1.56it/s]

Error for https://huggingface.co/lmms-lab/LLaVA-Video-72B-Qwen2: HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)


Scraping model cards:  77%|███████▋  | 806/1050 [09:23<02:38,  1.54it/s]

Error for https://huggingface.co/zelk12/MT3-Gen3-MU-gemma-2-Rv0.3MTM2MU-9B_DELETED: HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)


Scraping model cards:  78%|███████▊  | 818/1050 [09:40<02:43,  1.42it/s]

Error for https://huggingface.co/Steelskull/L3.3-MS-Nevoria-70b: HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)


Scraping model cards: 100%|██████████| 1050/1050 [12:48<00:00,  1.37it/s]

Scraping complete! Saved as 'huggingface_models_with_cards.csv'


### Images

In [6]:
# Input CSV file
input_csv = "huggingface_models_with_cards.csv"

# Folder to save images
output_folder = "huggingface_model_images"
os.makedirs(output_folder, exist_ok=True)

# Load CSV
df = pd.read_csv(input_csv)

# Detect column name for URLs
url_column = None
for c in df.columns:
    if "url" in c.lower():
        url_column = c
        break

if not url_column:
    raise ValueError("No column found with 'url' in its name!")

print(f"Using URL column: {url_column}")

# Loop through models
for index, row in df.iterrows():
    model_url = str(row[url_column])
    if not model_url.startswith("http"):
        model_url = "https://" + model_url

    model_name = str(row.get("Model_Name", f"model_{index}")).replace("/", "_")

    try:
        response = requests.get(model_url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        # Find all possible image tags
        img_tags = soup.find_all("img")

        found_any = False

        for i, img in enumerate(img_tags):
            img_url = img.get("src") or img.get("data-src")
            if not img_url:
                continue

            # Skip base64 or small icons
            if img_url.startswith("data:") or "favicon" in img_url or "emoji" in img_url:
                continue

            # Convert relative URLs to absolute
            img_url = urljoin(model_url, img_url)

            # Get image extension
            file_ext = img_url.split(".")[-1].split("?")[0]
            if len(file_ext) > 5:
                file_ext = "jpg"

            filename = f"{model_name}_{i}.{file_ext}"
            filepath = os.path.join(output_folder, filename)

            # Download image
            img_resp = requests.get(img_url, timeout=10)
            if img_resp.status_code == 200:
                with open(filepath, "wb") as f:
                    f.write(img_resp.content)
                print(f"Saved image: {filename}")
                found_any = True

        if not found_any:
            print(f"No images found for {model_name}")

        time.sleep(1)

    except Exception as e:
        print(f"Error for {model_url}: {e}")

print("Scraping complete! Check 'huggingface_model_images' folder.")


NameError: name 'os' is not defined